# Цели, тактики и стратегии в Z3

**Автор заданий:** владелец этого ноутбука.  
**Инструменты:** Python 3 и Z3Py.

Ниже собраны пять авторских учебных задач и их решения. В каждой задаче мы сначала преобразуем `Goal`, а затем выбираем решатель или стратегию. Такой подход полезен не только для получения ответа: он позволяет увидеть, *какую* формулу в действительности решает Z3.

In [ ]:
from z3 import *
from time import perf_counter

def show_pipeline(goal, tactic):
    """Применяет тактику и печатает получившиеся подцели."""
    result = tactic(goal)
    print(f'Подцелей: {len(result)}')
    for i, subgoal in enumerate(result, 1):
        print(f'--- подцель {i} ({len(subgoal)} формул) ---')
        print(subgoal)
    return result

def solve_goal(goal, tactic):
    """Готовит цель тактикой и решает все оставшиеся подцели."""
    subgoals = tactic(goal)
    models = []
    for subgoal in subgoals:
        s = Solver()
        s.add(subgoal.as_expr())
        if s.check() == sat:
            models.append(s.model())
    return models


## 1. Диагностика цели

**Задача.** Для ограничений над целыми числами построить конвейер `simplify → solve-eqs → smt`. Проследить, как равенство `z = x - y` устраняется до запуска общего SMT-решателя, и сравнить с непосредственным `smt`.

**Идея решения.** `simplify` нормализует локальные выражения, а `solve-eqs` подставляет переменные, определённые равенствами. Поэтому дорогому финальному решателю достаётся меньшая цель.

In [ ]:
x, y, z = Ints('x y z')
g1 = Goal()
g1.add(x + y == 10, x >= 0, y >= 0, z == x - y, z*z <= 4)

prepare_lia = Then('simplify', 'solve-eqs')
print('После подготовки:')
show_pipeline(g1, prepare_lia)

print('Модели после подготовки:', solve_goal(g1, prepare_lia))
print('Полная стратегия:', Then('simplify', 'solve-eqs', 'smt'))
print('Результат smt напрямую:', Tactic('smt')(g1))

## 2. Маршрутизатор формул

**Задача.** Сделать стратегию, которая распознаёт два частых фрагмента: кванторно-свободную линейную арифметику (`QF_LIA`) и кванторно-свободные битовые векторы (`QF_BV`). Для каждого фрагмента выбрать собственный конвейер; всё остальное отправить в универсальный `smt`.

**Идея решения.** Встроенные пробы Z3 (`Probe`) проверяют класс цели. `If` — это комбинатор тактик: он выбирает ветку в момент применения к `Goal`, а не при построении Python-программы.

In [ ]:
lia_branch = Then('simplify', 'solve-eqs', 'lia2pb', 'smt')
bv_branch = Then('simplify', 'solve-eqs', 'bit-blast', 'sat')
fallback_branch = Then('simplify', 'smt')

auto_route = If(Probe('is-qflia'), lia_branch,
                If(Probe('is-qfbv'), bv_branch, fallback_branch))

a, b = Ints('a b')
lia_goal = Goal(); lia_goal.add(a >= 0, b >= 0, 2*a + 3*b == 17)
p, q = BitVecs('p q', 8)
bv_goal = Goal(); bv_goal.add(p ^ q == 0x5A, UGT(p + q, BitVecVal(100, 8)))

for name, goal in [('QF_LIA', lia_goal), ('QF_BV', bv_goal)]:
    print(f'\n{name}:')
    show_pipeline(goal, auto_route)


## 3. Взломщик 8-битного протокола

**Задача.** Найти все пары байтов `a`, `b`, для которых `c = (a xor b) + 17`, `RotateLeft(a, 1) xor c = 0xA5`, а сумма `a + b` больше 200 в **беззнаковом** порядке. Затем заменить `UGT` на обычное `>` и объяснить разницу.

**Идея решения.** Для битовых векторов переполнение является частью семантики. `UGT` сравнивает байты как числа 0…255, тогда как `>` — как знаковые числа −128…127. Стратегия приводит формулу к булевой форме (`bit-blast`) и передаёт её SAT-решателю.

In [ ]:
a8, b8, c8 = BitVecs('a8 b8 c8', 8)
protocol = [
    c8 == (a8 ^ b8) + BitVecVal(17, 8),
    RotateLeft(a8, 1) ^ c8 == BitVecVal(0xA5, 8),
    UGT(a8 + b8, BitVecVal(200, 8)),
]
g3 = Goal(); g3.add(*protocol)
bv_strategy = Then('simplify', 'solve-eqs', 'bit-blast', 'sat')
show_pipeline(g3, bv_strategy)

def enumerate_models(constraints, limit=10):
    s = Solver(); s.add(*constraints)
    answer = []
    while len(answer) < limit and s.check() == sat:
        m = s.model()
        answer.append((m.eval(a8).as_long(), m.eval(b8).as_long(), m.eval(c8).as_long()))
        s.add(Or(a8 != m.eval(a8), b8 != m.eval(b8), c8 != m.eval(c8)))
    return answer

print('Первые модели (unsigned):', enumerate_models(protocol))
signed_protocol = protocol[:-1] + [a8 + b8 > BitVecVal(200, 8)]
print('Первые модели (signed):  ', enumerate_models(signed_protocol))

## 4. Мини-судоку 4×4

**Задача.** Закодировать Sudoku 4×4 и собрать стратегию, состоящую из повторяемой дешёвой подготовки и финального поиска. Сравнить почти заполненную доску и доску с минимумом подсказок.

**Идея решения.** `Repeat(Then(...), max=3)` ограниченно повторяет локальные преобразования; это важно, поскольку не каждая тактика умеет сама определить неподвижную точку. `OrElse` даёт запасной путь, если выбранная подготовка неприменима.

In [ ]:
cells = [[Int(f'cell_{r}_{c}') for c in range(4)] for r in range(4)]
sudoku_rules = []
for row in cells: sudoku_rules += [And([1 <= v for v in row]), And([v <= 4 for v in row]), Distinct(row)]
for c in range(4): sudoku_rules.append(Distinct([cells[r][c] for r in range(4)]))
for br in (0, 2):
    for bc in (0, 2):
        sudoku_rules.append(Distinct([cells[r][c] for r in range(br, br+2) for c in range(bc, bc+2)]))

givens = [cells[0][0] == 1, cells[0][3] == 4, cells[1][1] == 4, cells[2][2] == 4, cells[3][0] == 4]
g4 = Goal(); g4.add(*(sudoku_rules + givens))
sudoku_strategy = Then(Repeat(OrElse(Then('simplify', 'solve-eqs'), 'skip'), max=3), 'smt')
result = sudoku_strategy(g4)
print('Финальных подцелей:', len(result), '; состояние:', result)

s = Solver(); s.add(*(sudoku_rules + givens)); assert s.check() == sat
m = s.model()
for row in cells:
    print([m.eval(v).as_long() for v in row])

## 5. Портфель стратегий для бюджета

**Задача.** Выбрать объёмы трёх проектов в рамках бюджета и доступности сотрудников. Объёмы — целые числа, доступность кодируется битовой маской. Создать две стратегии: арифметическую и битовую, измерить их на одной цели и выбрать более подходящую по числу битовых операций.

**Идея решения.** Не существует одной лучшей последовательности тактик. Портфель — практичный способ объединить несколько эвристик: сначала измеряем формулу, потом выбираем путь. Для настоящей оптимизации обычно удобнее `Optimize`; здесь фокус именно на подготовке и проверке достижимости.

In [ ]:
p1, p2, p3 = Ints('p1 p2 p3')
team = BitVec('team', 4)  # один бит на сотрудника
budget_goal = Goal()
budget_goal.add(
    p1 >= 0, p2 >= 0, p3 >= 0,
    5*p1 + 7*p2 + 4*p3 <= 40,
    2*p1 + p2 >= 5,
    p3 <= p1 + 2,
    (team & BitVecVal(0b0011, 4)) == BitVecVal(0b0011, 4),
    Implies(p2 >= 2, (team & BitVecVal(0b1100, 4)) != 0),
)

arithmetic_first = Then('simplify', 'solve-eqs', 'propagate-values', 'smt')
# В смешанной цели bit-blast неприменим ко всей формуле: в ней есть Int.
# Поэтому битовая ветка делает безопасную пропагацию и передаёт комбинацию теорий SMT.
bitvector_first = Then('simplify', 'propagate-values', 'solve-eqs', 'smt')

def contains_bv(expr):
    if is_bv(expr): return True
    return any(contains_bv(child) for child in expr.children())

def choose_portfolio(goal):
    formulas = list(goal)
    bv_count = sum(contains_bv(f) for f in formulas)
    return bitvector_first if bv_count * 2 >= len(formulas) else arithmetic_first

for label, strategy in [('арифметическая', arithmetic_first), ('битовая', bitvector_first), ('выбранная', choose_portfolio(budget_goal))]:
    started = perf_counter()
    outcome = strategy(budget_goal)
    elapsed_ms = (perf_counter() - started) * 1000
    print(f'{label:16}: {len(outcome)} подцелей, {elapsed_ms:.2f} мс')

# Упражнение для продолжения: сгенерировать 20 вариантов budget_goal
# и заменить простое правило choose_portfolio измеренными порогами.

## Вывод

Формальный стиль мышления здесь означает: отделять модель от метода поиска, проверять форму цели после каждого существенного преобразования и выбирать алгоритм под структуру ограничений. Тактики — маленькие преобразования, комбинаторы задают управление, а стратегия превращает их в воспроизводимый способ решения класса задач.